# 5 . Simple 1m EMA Breakout (long-only)

**Rules**
- Timeframe: **1-minute**, long only.
- **Entry:** a candle forms *fully above* the 1m **EMA-200** (whole candle: low > EMA200).
- **Exit:** a candle forms *fully below* the 1m **EMA-50** (whole candle: high < EMA50).
- **Stop-loss:** the previous swing low (structure stop).

This maps directly onto `EmaCross`: `entry_mode='full_candle'` on EMA-200, `exit_mode='below_ema'`
with `exit_ema=50`, long-only, and a `ref_col` stop on the swing low.


## Setup
If you changed library code, restart the kernel so the latest `quant` loads.


In [ ]:
import sys, os, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..')); import quant
import pandas as pd
from quant.data import get_ohlcv
from quant.engine import BacktestConfig
from quant.strategies import EmaCross
from quant.reporting import print_summary
from quant.viz import ResearchChart
pd.set_option('display.max_rows', 200)

## 1 . Data (1-minute gold)
PAXGUSDT proxy by default; switch to true spot with `SYMBOL='XAUUSD', SOURCE='dukascopy'`.
A ~2-3 month window keeps the plot responsive and gives enough trades; widen later.


In [ ]:
SYMBOL, SOURCE, MARKET = 'PAXGUSDT', 'binance', 'spot'
START, END = '2026-05-01', '2026-07-22'
df = get_ohlcv(SYMBOL, '1m', start=START, end=END, source=SOURCE, market=MARKET, tz='UTC')
print(f'{len(df):,} 1m bars  {df.open_time.min()} -> {df.open_time.max()}')

## 2 . Strategy + config
`swings_left/right` set how local the 'previous low' is (5/5 = a 5-bar pivot). The stop sits at
that swing low; if it is ever unusable, it falls back to a 1% stop. Costs = your Exness spread (0.12).


In [ ]:
strat = EmaCross(
    ema_period=200, entry_mode='full_candle', confirm_n=1,     # full candle above EMA200
    allow_long=True, allow_short=False,                        # long only
    exit_mode='below_ema', exit_ema=50,                        # exit: full candle below EMA50
    swings_left=5, swings_right=5,                             # 'previous low' = 5-bar swing
)
cfg = BacktestConfig(
    initial_cash=10_000, spread=0.12, slippage_bps=1.0,
    exit_enabled=True,
    sl_mode='ref_col', sl_ref_long_col='swing_last_low',       # stop = previous swing low
    sl_buffer_pct=0.0, sl_fallback_mode='entry_pct', sl_fallback_value=1.0,
    tp_mode='none',                                            # no target; exit rule handles it
    sizing_mode='risk_pct_equity', sizing_value=1.0,           # risk 1% of equity per trade
    allow_rule_close=True,
)
res = strat.backtest(df, cfg)
print_summary(res, df=df)

## 3 . Trade table (with close reason)
Every trade: entry/exit time & price, the stop that was set, PnL, and *why* it closed
(`take_profit` n/a here; expect `signal` = EMA50 exit, `stop_loss` = swing-low stop, or
`forced_close_end`).


In [ ]:
cols = ['side','entry_time','entry_price','stop_price','exit_time','exit_price',
        'pnl','return_pct','bars_held','close_reason']
trades = res.trades[cols].copy()
trades['pnl'] = trades['pnl'].round(2); trades['return_pct'] = trades['return_pct'].round(2)
for c in ['entry_price','exit_price','stop_price']: trades[c] = trades[c].round(3)
print('close-reason counts:'); display(trades['close_reason'].value_counts())
trades

## 4 . Chart — candles + EMA200/EMA50 + entries/exits + stop-loss
Opens zoomed to the most recent bars (native 1m candles). **Click any legend item to toggle it**
(candles, EMA200, EMA50, long entry, exit win/loss, stop-loss, close line). Pan/zoom to explore;
hover a marker for full trade details.


In [ ]:
ch = ResearchChart(df, candles=True, initial_bars=400, title=f'{SYMBOL} 1m - EMA breakout')
ch.add_ema(200); ch.add_ema(50)
ch.add_trades(res.trades)          # entry/exit markers + dotted stop-loss lines
fig = ch.show()
fig

## 5 . Zoom straight to one trade
Jump the view to a chosen trade so you clearly see the candles at entry, the exit, and the stop.
(Change the index to inspect any trade from the table above.)


In [ ]:
tr = res.trades.sort_values('pnl').iloc[-1]     # the biggest winner; try .iloc[0] for worst
pad = pd.Timedelta(minutes=120)
x0 = pd.to_datetime(tr['entry_time']) - pad
x1 = pd.to_datetime(tr['exit_time']) + pad
fig.update_xaxes(range=[x0, x1])
print(f"{tr['side']} {tr['entry_time']} @ {tr['entry_price']:.3f} -> {tr['exit_time']} @ "
      f"{tr['exit_price']:.3f} | stop {tr['stop_price']:.3f} | pnl {tr['pnl']:.2f} | {tr['close_reason']}")
fig

## Notes
- **Toggling:** every layer is a legend entry — click to show/hide candles, either EMA, entry
  markers, winning/losing exits, or the stop-loss lines.
- **Stop-loss lines** are the dotted amber segments at each trade's stop (the previous swing low).
- **Costs matter:** `spread=0.12` is your Exness spread; raise it to stress-test.
- **Validate:** re-run on a different window and on true spot XAU/USD (`source='dukascopy'`) before
  trusting the numbers. On 1m, cost drag is heavy — watch profit factor and max drawdown, not just return.
